# Practice 5-4. Neo4j 기반 GraphRAG 구현 — 지식 그래프 저장

책은 Neo4j Aura(클라우드)에 인스턴스를 새로 만들고, `02-graphrag-knowledge-graph-indexing.ipynb`에서 만든 GraphRAG 산출물(parquet)을 업로드합니다.
이 노트북은 `rag-master/docker-compose.yml`의 **로컬 Neo4j 컨테이너**(`neo4j:5.12.0`, APOC 플러그인
포함)를 그대로 사용합니다. 이 컨테이너는 `unsloth_studio`와 동일한 외부 Docker 네트워크(`linker`)에 붙어 있어
`bolt://neo4j:7687`로 바로 접속할 수 있습니다.

> **공유 인스턴스 주의**: 이 Neo4j는 다른 프로젝트와 함께 쓰는 공용 인스턴스라서 이미 다른 라벨의 데이터가
> 들어 있을 수 있습니다. GraphRAG는 원래부터 `__Document__`, `__Chunk__`, `__Entity__`, `__Community__`처럼
> 이중 언더스코어로 감싼 라벨을 쓰므로 다른 프로젝트의 라벨과 충돌할 가능성은 낮지만, 이 노트북의 재실행 안전
> 정리(2단계) 역시 **이 라벨들만** 대상으로 삭제합니다.

> **스키마 차이 주의**: 책이 쓰는 GraphRAG 버전은 `create_final_nodes.parquet`(엔티티별 `community`/`x`/`y` 좌표
> 포함)를 별도로 출력하지만, 현재 설치된 GraphRAG 3.x는 그래프 임베딩·좌표 스냅숏이 기본적으로 꺼져 있어
> (`snapshots.graphml/embeddings: false`) 이 파일 자체가 없습니다. 대신 엔티티의 커뮤니티 소속은
> `communities.parquet`의 `entity_ids` 목록에서 직접 끌어옵니다. 또한 파일명에 `create_final_` 접두사가 없고,
> `text_units.parquet`의 문서 참조 컬럼도 책의 `document_ids`(리스트)가 아니라 단수형 `document_id`입니다.

## 0. 환경 설정

In [1]:
!pip install neo4j

In [2]:
import os
import time
from pathlib import Path

import pandas as pd
from neo4j import GraphDatabase

# --- docker-compose.yml의 Neo4j 연결 정보 ---
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://neo4j:7687")
NEO4J_AUTH = os.environ["NEO4J_AUTH"]
NEO4J_USERNAME, NEO4J_PASSWORD = NEO4J_AUTH.split("/", 1)
NEO4J_DATABASE = "neo4j"

# --- 02-graphrag-knowledge-graph-indexing.ipynb에서 구축한 GraphRAG 산출물 ---
GRAPHRAG_FOLDER = Path("working_directory/output")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Neo4j 연결 성공:", NEO4J_URI)

Neo4j 연결 성공: bolt://neo4j:7687


## 1. 배치 임포트 함수

책과 동일하게, 데이터프레임을 배치로 나누어 `UNWIND`로 벌크 적재하는 헬퍼 함수를 정의합니다.

In [3]:
def batched_import(statement, df, batch_size=1000):
    """
    Import a dataframe into Neo4j using a batched approach.

    Parameters: statement is the Cypher query to execute, df is the dataframe to
    import, and batch_size is the number of rows to import in each batch.
    """
    total = len(df)
    start_s = time.time()
    for start in range(0, total, batch_size):
        batch = df.iloc[start : min(start + batch_size, total)]
        result = driver.execute_query(
            "UNWIND $rows AS value " + statement,
            rows=batch.to_dict("records"),
            database_=NEO4J_DATABASE,
        )
        print(result.summary.counters)
    print(f"{total} rows in {time.time() - start_s} s.")
    return total

## 2. 재실행 안전 — 기존 GraphRAG 라벨만 정리

공유 Neo4j 인스턴스이므로 전체 DB를 비우지 않고, 이 노트북이 만드는 라벨(`__Document__`, `__Chunk__`,
`__Entity__`, `__Community__`, `Finding`)에 해당하는 노드만 지운 뒤 다시 적재합니다.

In [4]:
GRAPHRAG_LABELS = ["__Document__", "__Chunk__", "__Entity__", "__Community__", "Finding"]

for label in GRAPHRAG_LABELS:
    result = driver.execute_query(
        f"MATCH (n:`{label}`) DETACH DELETE n RETURN count(n) AS deleted",
        database_=NEO4J_DATABASE,
    )
    print(label, "->", result.records[0]["deleted"] if result.records else 0, "삭제")

__Document__ -> 0 삭제
__Chunk__ -> 0 삭제
__Entity__ -> 0 삭제
__Community__ -> 0 삭제
Finding -> 0 삭제


## 3. 제약 조건 설정

중복 데이터를 방지하기 위한 유일성 제약을 Cypher로 설정합니다.

In [5]:
statements = [
    "create constraint chunk_id if not exists for (c:__Chunk__) require c.id is unique",
    "create constraint document_id if not exists for (d:__Document__) require d.id is unique",
    "create constraint community_id if not exists for (c:__Community__) require c.community is unique",
    "create constraint entity_id if not exists for (e:__Entity__) require e.id is unique",
    "create constraint entity_title if not exists for (e:__Entity__) require e.name is unique",
    "create constraint related_id if not exists for ()-[rel:RELATED]->() require rel.id is unique",
]

for statement in statements:
    if len((statement or "").strip()) > 0:
        print(statement)
        driver.execute_query(statement, database_=NEO4J_DATABASE)

create constraint chunk_id if not exists for (c:__Chunk__) require c.id is unique
create constraint document_id if not exists for (d:__Document__) require d.id is unique
create constraint community_id if not exists for (c:__Community__) require c.community is unique
create constraint entity_id if not exists for (e:__Entity__) require e.id is unique
create constraint entity_title if not exists for (e:__Entity__) require e.name is unique
create constraint related_id if not exists for ()-[rel:RELATED]->() require rel.id is unique


## 4. 지식 그래프 구축

먼저 최종 문서를 DB에 업로드합니다. `documents.parquet`에서 문서 데이터를 가져와 `__Document__` 노드를 생성합니다.

In [6]:
doc_df = pd.read_parquet(GRAPHRAG_FOLDER / "documents.parquet", columns=["id", "title"])

statement = """
MERGE (d:__Document__ {id:value.id})
SET d += value {.title}
"""

batched_import(statement, doc_df)

SummaryCounters{labels_added: 1, nodes_created: 1, properties_set: 2, contains_updates: True, contains_system_updates: False}
1 rows in 0.313615083694458 s.


1

다음으로, 텍스트 유닛(청크)을 DB에 업로드합니다. `text_units.parquet`에서 텍스트 유닛 데이터를 로드하여
`__Chunk__` 노드를 생성하고 문서와 연결합니다. 책은 이 컬럼이 리스트(`document_ids`)라 `UNWIND`가 필요하지만,
현재 버전은 청크 하나당 문서 하나(`document_id`, 단수)라 바로 `MATCH`하면 됩니다.

In [7]:
text_df = pd.read_parquet(
    GRAPHRAG_FOLDER / "text_units.parquet", columns=["id", "text", "n_tokens", "document_id"]
)

statement = """
MERGE (c:__Chunk__ {id:value.id})
SET c += value {.text, .n_tokens}
WITH c, value
MATCH (d:__Document__ {id: value.document_id})
MERGE (c)-[:PART_OF]->(d)
"""

batched_import(statement, text_df)

SummaryCounters{labels_added: 29, relationships_created: 29, nodes_created: 29, properties_set: 87, contains_updates: True, contains_system_updates: False}
29 rows in 0.3904073238372803 s.


29

이제 엔티티를 DB에 업로드합니다. `entities.parquet`에서 엔티티 데이터를 가져와 `__Entity__` 노드를 생성하고
관련 속성을 추가한 뒤, 텍스트 청크와 연결합니다.

In [8]:
entity_df = pd.read_parquet(
    GRAPHRAG_FOLDER / "entities.parquet",
    columns=["title", "type", "description", "human_readable_id", "id", "text_unit_ids"],
)

statement = """
MERGE (e:__Entity__ {id: value.id})
SET e.human_readable_id = value.human_readable_id,
    e.description = value.description,
    e.name = coalesce(replace(value.title, '\"', ''), 'Unknown')
WITH e, value
CALL apoc.create.addLabels(e, CASE WHEN coalesce(value.type, "") = "" THEN [] ELSE
    [apoc.text.upperCamelCase(replace(value.type, '\"', ''))] END) YIELD node
UNWIND value.text_unit_ids AS text_unit
MATCH (c:__Chunk__ {id: text_unit})
MERGE (c)-[:HAS_ENTITY]->(e)
"""

batched_import(statement, entity_df)

SummaryCounters{labels_added: 157, relationships_created: 185, nodes_created: 157, properties_set: 628, contains_updates: True, contains_system_updates: False}
157 rows in 0.6986682415008545 s.


157

이어서 관계를 DB에 업로드합니다. `relationships.parquet`에서 관계 데이터를 로드하여 엔티티 간 `RELATED` 관계를
생성합니다.

In [9]:
rel_df = pd.read_parquet(
    GRAPHRAG_FOLDER / "relationships.parquet",
    columns=["source", "target", "id", "combined_degree", "weight", "human_readable_id",
             "description", "text_unit_ids"],
)
rel_df = rel_df.rename(columns={"combined_degree": "rank"})

rel_statement = """
    MATCH (source:__Entity__ {name:replace(value.source,'\"','')})
    MATCH (target:__Entity__ {name:replace(value.target,'\"','')})
    MERGE (source)-[rel:RELATED {id: value.id}]->(target)
    SET rel += value {.rank, .weight, .human_readable_id, .description, .text_unit_ids}
    RETURN count(*) as createdRels
"""

batched_import(rel_statement, rel_df)

SummaryCounters{relationships_created: 131, properties_set: 786, contains_updates: True, contains_system_updates: False}
131 rows in 0.3562009334564209 s.


131

다음으로 커뮤니티를 DB에 업로드합니다. `communities.parquet`에서 커뮤니티 데이터를 로드하여 `__Community__`
노드를 생성합니다. 책은 별도의 `create_final_nodes.parquet`(엔티티별 `community` 컬럼)로 엔티티-커뮤니티
연결을 만들지만, 이 파일이 없는 현재 버전에서는 `communities.parquet`가 이미 담고 있는 `entity_ids`
목록에서 바로 연결을 만듭니다(관계를 통한 간접 연결도 belt-and-suspenders로 함께 유지합니다).

In [10]:
community_df = pd.read_parquet(
    GRAPHRAG_FOLDER / "communities.parquet",
    columns=["id", "community", "level", "title", "entity_ids", "relationship_ids", "text_unit_ids"],
)

statement = """
MERGE (c:__Community__ {community: value.community})
SET c.id = value.id,
    c.level = value.level,
    c.title = value.title
WITH c, value
UNWIND value.text_unit_ids AS text_unit_id
MATCH (t:__Chunk__ {id: text_unit_id})
MERGE (c)-[:HAS_CHUNK]->(t)
WITH distinct c, value
UNWIND value.entity_ids AS entity_id
MATCH (e:__Entity__ {id: entity_id})
MERGE (e)-[:IN_COMMUNITY]->(c)
WITH distinct c, value
UNWIND value.relationship_ids AS rel_id
MATCH (start:__Entity__)-[:RELATED {id: rel_id}]->(end:__Entity__)
MERGE (start)-[:IN_COMMUNITY]->(c)
MERGE (end)-[:IN_COMMUNITY]->(c)
RETURN count(distinct c) as createdCommunities
"""

batched_import(statement, community_df)

SummaryCounters{labels_added: 14, relationships_created: 120, nodes_created: 14, properties_set: 56, contains_updates: True, contains_system_updates: False}
14 rows in 0.504899263381958 s.


14

이제 커뮤니티 리포트를 DB에 반영합니다. `community_reports.parquet` 파일에서 커뮤니티 보고서를 가져와
커뮤니티 노드에 속성을 추가하고, 발견 사항(findings)을 개별적인 `Finding` 노드로 반영합니다. 책은
`rank_explanation` 컬럼을 쓰지만, 현재 버전은 `rating_explanation`입니다.

In [11]:
community_report_df = pd.read_parquet(
    GRAPHRAG_FOLDER / "community_reports.parquet",
    columns=["id", "community", "level", "title", "summary", "findings", "rank",
             "rating_explanation", "full_content"],
)

community_statement = """
MERGE (c:__Community__ {community: value.community})
SET c.level = value.level,
    c.title = value.title,
    c.rank = value.rank,
    c.rating_explanation = value.rating_explanation,
    c.full_content = value.full_content,
    c.summary = value.summary
WITH c, value
UNWIND range(0, size(value.findings)-1) AS finding_idx
WITH c, value, finding_idx, value.findings[finding_idx] AS finding
MERGE (f:Finding {id: toString(value.community) + ':' + toString(finding_idx)})
SET f += finding
MERGE (c)-[:HAS_FINDING]->(f)
"""

batched_import(community_statement, community_report_df)

SummaryCounters{labels_added: 78, relationships_created: 78, nodes_created: 78, properties_set: 318, contains_updates: True, contains_system_updates: False}
14 rows in 0.2671689987182617 s.


14

## 5. 적재 결과 검증

In [12]:
result = driver.execute_query(
    """
    MATCH (n)
    WHERE any(l IN labels(n) WHERE l IN ['__Document__','__Chunk__','__Entity__','__Community__','Finding'])
    UNWIND labels(n) AS label
    WITH label, count(*) AS cnt
    WHERE label IN ['__Document__','__Chunk__','__Entity__','__Community__','Finding']
    RETURN label, cnt ORDER BY label
    """,
    database_=NEO4J_DATABASE,
)
for record in result.records:
    print(record["label"], ":", record["cnt"])

rel_result = driver.execute_query(
    "MATCH ()-[r]->() WHERE type(r) IN ['PART_OF','HAS_ENTITY','RELATED','HAS_CHUNK','IN_COMMUNITY','HAS_FINDING'] "
    "RETURN type(r) AS rel_type, count(*) AS cnt ORDER BY rel_type",
    database_=NEO4J_DATABASE,
)
for record in rel_result.records:
    print(record["rel_type"], ":", record["cnt"])

Finding : 78
__Chunk__ : 29
__Community__ : 14
__Document__ : 1
__Entity__ : 157
HAS_CHUNK : 36
HAS_ENTITY : 185
HAS_FINDING : 78
IN_COMMUNITY : 84
PART_OF : 29
RELATED : 131


이 과정을 통해 GraphRAG의 결과물을 로컬 Neo4j에 저장하여 실시간 질의나 분석에 활용 가능한 지식 그래프
데이터베이스를 완성할 수 있습니다. 구글 드라이브나 Neo4j Aura 클라우드 없이도, 이 워크스페이스와 같은
Docker 네트워크에 떠 있는 로컬 Neo4j만으로 책과 동일한 그래프 DB를 구축했습니다.

다음 노트북(`05-neo4j-local-global-graph-rag.ipynb`)에서 이 Neo4j 그래프 DB를 대상으로 로컬 검색과 글로벌 검색을
직접 구현합니다.